# Catnip — Consolidate Stage 1 v2 Archive on Cloudflare R2

This notebook downloads all 46 split chunks (`compressed/stage1_v2_sliced.tar.part_*`), streams and concatenates them sequentially into a single unified archive `/content/stage1_v2_sliced.tar` (~13.86 GB), uploads the unified archive to Cloudflare R2 via S3 Multipart Upload (bypassing Wrangler's 300 MiB per-object limit), and safely deletes the 46 split chunks from R2.

**Hardware:** Standard CPU-only Colab runtime (no GPU needed).

### 0. Install Dependencies

In [ ]:
!pip install --quiet boto3 tqdm
print("Dependencies installed successfully!")

### 1. Configure R2 Credentials & S3 Client

Add `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` to Colab Secrets (the key-lock icon on the left panel), or enter them when prompted.

In [ ]:
import os
import sys
import getpass
import boto3

try:
    from google.colab import userdata
    r2_access_key = userdata.get("R2_ACCESS_KEY_ID")
    r2_secret_key = userdata.get("R2_SECRET_ACCESS_KEY")
except Exception:
    r2_access_key = os.environ.get("R2_ACCESS_KEY_ID")
    r2_secret_key = os.environ.get("R2_SECRET_ACCESS_KEY")

if not r2_access_key:
    r2_access_key = getpass.getpass("Enter R2_ACCESS_KEY_ID: ")
if not r2_secret_key:
    r2_secret_key = getpass.getpass("Enter R2_SECRET_ACCESS_KEY: ")

R2_ENDPOINT = "https://eccb0b608bd926e793224944aba32cf4.r2.cloudflarestorage.com"
BUCKET = "catnip-canonical"
UNIFIED_KEY = "compressed/stage1_v2_sliced.tar"
PART_PREFIX = UNIFIED_KEY + ".part_"

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=r2_access_key,
    aws_secret_access_key=r2_secret_key,
    region_name="auto",
)

# Connectivity test
resp = s3.list_objects_v2(Bucket=BUCKET, MaxKeys=1)
print(f"Connected to Cloudflare R2 bucket '{BUCKET}' successfully!")

### 2. Scan Split Parts on R2

In [ ]:
paginator = s3.get_paginator("list_objects_v2")
parts = []
for page in paginator.paginate(Bucket=BUCKET, Prefix=PART_PREFIX):
    for item in page.get("Contents", []):
        parts.append(item)

parts.sort(key=lambda p: p["Key"])
total_expected_bytes = sum(p["Size"] for p in parts)

print(f"Found {len(parts)} split parts in R2:")
print(f"  First part : {parts[0]['Key'] if parts else 'None'}")
print(f"  Last part  : {parts[-1]['Key'] if parts else 'None'}")
print(f"  Total size : {total_expected_bytes:,} bytes ({total_expected_bytes / 1e9:.2f} GB)")

assert len(parts) > 0, f"No split parts found for prefix: {PART_PREFIX}"

### 3. Stream-Download and Concatenate into Single Archive

Streams each chunk directly into `/content/stage1_v2_sliced.tar` sequentially to minimize disk usage.

In [ ]:
import shutil
from pathlib import Path
from tqdm.auto import tqdm

free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Colab /content free disk: {free_gb:.1f} GB")
assert free_gb >= 18.0, f"Insufficient disk space: {free_gb:.1f} GB < 18 GB required."

LOCAL_TAR = Path("/content/stage1_v2_sliced.tar")
if LOCAL_TAR.exists():
    LOCAL_TAR.unlink()

bytes_written = 0
with open(LOCAL_TAR, "wb") as out_f:
    with tqdm(total=total_expected_bytes, unit="B", unit_scale=True, unit_divisor=1024, desc="Downloading & joining") as pbar:
        for idx, part in enumerate(parts):
            key = part["Key"]
            resp = s3.get_object(Bucket=BUCKET, Key=key)
            body = resp["Body"]
            while True:
                chunk = body.read(16 * 1024 * 1024)  # 16 MB chunk buffer
                if not chunk:
                    break
                out_f.write(chunk)
                bytes_written += len(chunk)
                pbar.update(len(chunk))

print(f"\nDownload complete: {LOCAL_TAR} ({LOCAL_TAR.stat().st_size:,} bytes)")
assert LOCAL_TAR.stat().st_size == total_expected_bytes, "Byte count mismatch!"

### 4. Verify Local Archive Integrity

In [ ]:
import tarfile

print(f"Verifying tar structure for {LOCAL_TAR}...")
with tarfile.open(LOCAL_TAR, "r") as tar:
    first_members = []
    for i, member in enumerate(tar):
        first_members.append(member.name)
        if i >= 10:
            break

assert len(first_members) > 0, "Tar file is empty or corrupted!"
print("Tar integrity check PASSED! Sample entries:")
for name in first_members[:5]:
    print(f"  - {name}")

### 5. Upload Unified Archive to R2 (S3 Multipart Upload)

Bypasses Wrangler's 300 MiB limit using `boto3` TransferConfig with 64 MB multipart chunks.

In [ ]:
from boto3.s3.transfer import TransferConfig

file_size = LOCAL_TAR.stat().st_size
config = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,
    multipart_chunksize=64 * 1024 * 1024,
    max_concurrency=8,
    use_threads=True,
)

with tqdm(total=file_size, unit="B", unit_scale=True, unit_divisor=1024, desc="Uploading unified tar") as pbar:
    def upload_callback(bytes_amount):
        pbar.update(bytes_amount)

    print(f"Uploading {LOCAL_TAR} ({file_size / 1e9:.2f} GB) to s3://{BUCKET}/{UNIFIED_KEY}...")
    s3.upload_file(
        Filename=str(LOCAL_TAR),
        Bucket=BUCKET,
        Key=UNIFIED_KEY,
        Config=config,
        Callback=upload_callback,
    )

print(f"\nUpload to s3://{BUCKET}/{UNIFIED_KEY} complete!")

### 6. Verify Remote Object & Safely Delete Split Parts

In [ ]:
# Strict verification of remote object
head_resp = s3.head_object(Bucket=BUCKET, Key=UNIFIED_KEY)
remote_size = head_resp.get("ContentLength", 0)
print(f"Remote unified object verified: s3://{BUCKET}/{UNIFIED_KEY}")
print(f"  Remote size: {remote_size:,} bytes")
print(f"  Local size : {file_size:,} bytes")
print(f"  ETag       : {head_resp.get('ETag', 'N/A')}")

assert remote_size == file_size, f"Remote size mismatch: {remote_size} != {file_size}"

# Atomic deletion of all split parts
print(f"\nSafely deleting {len(parts)} split parts from R2...")
batch_size = 1000
deleted_count = 0
for i in range(0, len(parts), batch_size):
    batch = parts[i : i + batch_size]
    delete_spec = {"Objects": [{"Key": p["Key"]} for p in batch]}
    del_resp = s3.delete_objects(Bucket=BUCKET, Delete=delete_spec)
    deleted_count += len(del_resp.get("Deleted", []))

print(f"Deleted {deleted_count} split part objects from R2.")

# Cleanup local disk
LOCAL_TAR.unlink()
print(f"Removed temporary local file {LOCAL_TAR}.")

print("\n" + "=" * 60)
print(f" SUCCESS: s3://{BUCKET}/{UNIFIED_KEY} is now canonical ({remote_size / 1e9:.2f} GB).")
print(" All split parts have been safely replaced.")
print("=" * 60)